## *IMPORTS*

In [0]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [0]:
cd /content/gdrive/My\ Drive/landmarks

/content/gdrive/My Drive/landmarks


In [0]:
!pip install git+git://github.com/friggog/dlib.git@sheep

  Cloning git://github.com/friggog/dlib.git (to revision sheep) to /tmp/pip-req-build-7twkjtdx
  Running command git clone -q git://github.com/friggog/dlib.git /tmp/pip-req-build-7twkjtdx
  Running command git checkout -b sheep --track origin/sheep
  Switched to a new branch 'sheep'
  Branch 'sheep' set up to track remote branch 'sheep' from 'origin'.
ERROR: Operation cancelled by user


In [0]:
!pip install 'menpo<0.9,>=0.8'

In [0]:
!pip install git+git://github.com/friggog/menpofit.git@sheep
  
!pip install menpowidgets
!pip install opencv-python

  Cloning git://github.com/friggog/menpofit.git (to revision sheep) to /tmp/pip-req-build-wonv3tiw
  Running command git clone -q git://github.com/friggog/menpofit.git /tmp/pip-req-build-wonv3tiw
  Running command git checkout -b sheep --track origin/sheep
  Switched to a new branch 'sheep'
  Branch 'sheep' set up to track remote branch 'sheep' from 'origin'.
  Stored in directory: /tmp/pip-ephem-wheel-cache-2hl82fiv/wheels/68/f8/0b/c4c795bc92503bcc03d0d754318a8cf707f99dff18acfc05be
Successfully built menpofit


In [0]:
#%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import random
import os
import cv2
import glob
import time
from IPython.core.display import clear_output
from IPython import display
import menpo
import menpofit
import menpo.io as mio
from menpowidgets import visualize_images
from menpo.transform import AlignmentSimilarity
from menpo.shape import bounding_box, mean_pointcloud
import dlib        
from menpofit.error import euclidean_error, euclidean_bb_normalised_error
from menpo.landmark.base import LandmarkGroup
from functools import partial
from menpo.base import LazyList
import menpo.io as mio
from sklearn.cluster import AffinityPropagation, MeanShift, KMeans
from menpo.shape import PointCloud
#from menpo.feature import vector_128_dsift, hellinger_vector_128_dsift
from menpofit.dlib.fitter import DlibERT, DlibWrapper
from sklearn.svm import LinearSVC, SVC
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import RidgeCV
from menpo.shape import PointCloud
from sklearn.decomposition import PCA
from shapes import FULL_THREED_SHAPE
from menpofit.builder import compute_reference_shape
from menpofit.error import euclidean_bb_normalised_error

"""
To check the classes in a module
=================================
import inspect
for name, obj in inspect.getmembers(menpofit.dlib):
    if inspect.isclass(obj):
        print(obj)

"""


ABS_PATH = os.path.join(os.getcwd(), 'absolute_yaw') 

## *LOAD IMAGES AND CROSS-VALIDATION*

### Functions

In [0]:
def sort_files(l):
    def f(path):
        path = path.split('/')[-1]
        path = path.split('.')[0]
        split = path.split('_')
        return int(split[0])
    
    return sorted(l, key=f)

def sorted_image_import(pattern, shuffle=False):
    file_list = sort_files(glob.glob(pattern + '/*.png'))
    if shuffle:
        random.shuffle(file_list)
    """
    The default behaviour of import_images() imports and attaches the landmark 
    files that share the same filename stem as the images. The group name of the 
    attached landmarks is based on the extension of the landmark file. This means 
    that it is much more convenient to locate the landmark files in the same directory 
    as the image files - LANDMARK FILE, SAME FOLDER *.PTS
    """
    l = LazyList([partial(mio.import_image, f) for f in file_list])
    print(len(l), 'images imported from', pattern)
    return l, file_list


def sorted_landmark_import(pattern, shuffle=False):
    file_list = sort_files(glob.glob(pattern + '/*.pts'))
    if shuffle:
        random.shuffle(file_list)
    l = LazyList([partial(mio.import_landmark_file, f) for f in file_list])
    print(len(l), 'landmarks imported from', pattern)
    return l, file_list  


def cross_validation_sets(lazy_list, file_list, k=5, verbose=False):

    n_raw_images = len(file_list)
    n_raw_per_fold = n_raw_images // k
    print('n_raw_per_fold: ', n_raw_per_fold)
    out_data = []
    
    for fold in range(k):
        
        test_start = fold * n_raw_per_fold
        test_end = (fold + 1) * n_raw_per_fold
        test_data = lazy_list[test_start:test_start + n_raw_per_fold]
        
        train_data = lazy_list[:test_start] + lazy_list[test_start + n_raw_per_fold:]
        out_data.append((train_data, test_data))
        
        if verbose:
            print('Fold', fold, len(test_data), 'test and', len(train_data), 'train')
    return out_data

### Run

In [0]:
path_to_images = os.path.join(ABS_PATH,'0_30')
images_30, files_30 = sorted_image_import(path_to_images) 
landmarks_30 = images_30.map(lambda x: x.landmarks) #Extracts the landmarks but they are still 

cv_images_30 = cross_validation_sets(images_30, files_30, verbose=True) 
cv_landmarks_30 = cross_validation_sets(landmarks_30, files_30)

879 images imported from /content/gdrive/My Drive/landmarks/absolute_yaw/0_30
n_raw_per_fold:  175
Fold 0 175 test and 704 train
Fold 1 175 test and 704 train
Fold 2 175 test and 704 train
Fold 3 175 test and 704 train
Fold 4 175 test and 704 train
n_raw_per_fold:  175


In [0]:
path_to_images = os.path.join(ABS_PATH,'30_60')
images_60, files_60 = sorted_image_import(path_to_images) 
landmarks_60 = images_60.map(lambda x: x.landmarks) #Extracts the landmarks but they are still 

cv_images_60 = cross_validation_sets(images_60, files_60, verbose=True) 
cv_landmarks_60 = cross_validation_sets(landmarks_60, files_60)

107 images imported from /content/gdrive/My Drive/landmarks/absolute_yaw/30_60
n_raw_per_fold:  21
Fold 0 21 test and 86 train
Fold 1 21 test and 86 train
Fold 2 21 test and 86 train
Fold 3 21 test and 86 train
Fold 4 21 test and 86 train
n_raw_per_fold:  21


In [0]:
path_to_images = os.path.join(ABS_PATH,'60_90')
images_90, files_90 = sorted_image_import(path_to_images) 
landmarks_90 = images_90.map(lambda x: x.landmarks) #Extracts the landmarks but they are still 

cv_images_90 = cross_validation_sets(images_90, files_90, verbose=True) 
cv_landmarks_90 = cross_validation_sets(landmarks_90, files_90)

89 images imported from /content/gdrive/My Drive/landmarks/absolute_yaw/60_90
n_raw_per_fold:  17
Fold 0 17 test and 72 train
Fold 1 17 test and 72 train
Fold 2 17 test and 72 train
Fold 3 17 test and 72 train
Fold 4 17 test and 72 train
n_raw_per_fold:  17


# *DlibERT - Ensemble of Regression Trees*
=====================================================================================================================
This technique utilize simple and fast feature (pixel intensities differences) 
to directly estimate the landmark positions. These estimated positions are 
subsequently refined with an iterative process done by a cascade of regressors. 
The regressors produces a new estimate from the previous one, trying to reduce 
the alignment error of the estimated points at each iteration.

Arguments:
- images (list of menpo.image.Image): The list of training images
- scales (float or tuple of float, optional): The scale value of each scale. They must provided in ascending order, i.e. from lowest to highest scale. If float, then a single scale is assumed. 
- verbose (bool, optional): If True, then the progress of building ERT will be printed.
- n_perturbations:(int or None, optional) – The number of perturbations to be generated from each of the bounding boxes using perturb_from_gt_bounding_box. Note that the total number of perturbations is n_perturbations * n_dlib_perturbations.
    
More info: https://menpofit.readthedocs.io/en/stable/api/menpofit/dlib/DlibERT.html

### Auxiliar Functions

In [0]:
def printb(*args, **kwargs):
    s_args = []
    for a in args:
        s_args.append(str(a))
    print('\033[1m'+' '.join(s_args)+'\033[0m', **kwargs)
    
def kmeans_for_landmark_set(landmarks, k=10):
    np_lm = get_np_lms(landmarks)
    km = KMeans(k)
    km.fit(np_lm)
    out = [PointCloud(x.reshape((-1, 2))) for x in km.cluster_centers_]
    return out

def mpi_svm(fitter, train, test, initialisations):
    train_poses, train_features = get_gt_poses(fitter, train, initialisations)
    test_poses, test_features = get_gt_poses(fitter, test, initialisations)
#     pca = PCA(n_components=256)
#     pca_train_features = pca.fit_transform(train_features)
#     pca_test_features = pca.transform(test_features)
    clf = LinearSVC(class_weight='balanced', C=100)
    clf.fit(train_features, train_poses)
    print('Clf Acc', clf.score(test_features, test_poses), ' ' * 40)
    p_init = clf.predict(test_features)
    pred_pc = [initialisations[p] for p in p_init]
    return pred_pc

def test_eval(fitter, images, initialisations=None, mean_conv=0, use_fitter=True, verbose=False, plme=False):
    error = 0
    errors = []
    plm_errors = []
    for i, image in enumerate(images):
        if verbose:
            print('Tested', i+1, end='\r')
        if initialisations is not None:
            if len(initialisations) == len(images):
                # provided a predicted good initialisation for each image so use that directly
#                 plt.clf()
#                 image.view()
#                 align_dec_lms_to_bb(initialisations[i], image.landmarks['PTS'].lms.bounding_box()).view()
#                 display.display(plt.gcf())
#                 display.clear_output(wait=True)
#                 input()
                if use_fitter:
                    result = fitter.fit_from_shape(image, 
                                            align_dec_lms_to_bb(initialisations[i], 
                                                                image.landmarks['PTS'].bounding_box()), 
                                            gt_shape=image.landmarks['PTS'])
                    e = result.final_error()
                else:
                    e = euclidean_bb_normalised_error(align_dec_lms_to_bb(initialisations[i], 
                                                                image.landmarks['PTS'].bounding_box()),
                                                      image.landmarks['PTS'])
                error += e
                errors.append(e)
            else:
                # prvided multiple candidate initialisations so fit using all intialisations and take lowest error
                # i.e. pose initialisation oracle
                results = []
                for init in initialisations:
                    result = fitter.fit_from_shape(image, 
                                        align_dec_lms_to_bb(init, image.landmarks['PTS'].bounding_box()), 
                                        gt_shape=image.landmarks['PTS'])
                    results.append(result)
                if mean_conv > 0:
                    final_shape = best_res_from_med(results, mean_conv)
#                     final_shape = best_res_min_dist_to_init(results, initialisations, mean_conv)
#                     result = fitter.fit_from_shape(image, final_shape, gt_shape=image.landmarks['PTS'].lms)
                    e = euclidean_bb_normalised_error(final_shape, image.landmarks['PTS'])
#                     e = result.final_error()
                else:
                    e = min([r.final_error() for r in results])
                    best_init = np.argmin([r.final_error() for r in results])
                
#                     plt.clf()
#                     image.view()
#                     align_dec_lms_to_bb(initialisations[best_init], image.landmarks['PTS'].lms.bounding_box()).view()
#                     display.display(plt.gcf())
#                     display.clear_output(wait=True)
#                     input()
                    
                error += e   
                errors.append(e)
        else:
            # just use the bounding box (i.e. mean shape) to initialise
            result = fitter.fit_from_bb(image, 
                                        image.landmarks['PTS'].bounding_box(), 
                                        gt_shape=image.landmarks['PTS'])
            error += result.final_error()
            errors.append(result.final_error())
            if plme:
                plm_errors.append(per_lm_errors(result.final_shape, image.landmarks['PTS']))
    if verbose:
        print()
    if plme:
        return error / len(errors), errors, plm_errors
    else:
        return error / len(errors), errors

In [0]:
def error_dist(errors, export=None):
    errors = sorted(errors)
    ys = []
    xs = np.arange(0, 1, 0.005)
    out_string = ''
    for x in xs:
        y = len(np.where(errors < x)[0]) / len(errors)
        ys.append(y)
        out_string += str(x) + ',' + str(y) + '\n'
    ys = np.array(ys)
    print('Mean error:         ', np.mean(errors))
    print('Less than 10% error:', len(np.where(np.array(errors) < 0.1)[0]) / len(errors))
    print('AUC:                ', np.mean(ys))
    if export is not None:
        with open(export + '.dat', 'w') as out_file:
            out_file.write(out_string)
    return (xs, ys)

def plot_errors(err_list, export=None):
    for (name, err) in err_list:
        printb(name.upper())
        (x, y) = error_dist(err, (export + '-' + name if export is not None else None))
        plt.plot(x, y, label=name)
        print()
    plt.legend(loc="lower right")
    plt.xlim((0, 0.2))
    plt.show()

### Main Function

In [0]:
def ERT(MPI=False, TIF=False, init_f=kmeans_for_landmark_set, init_n=10, 
        verbose=False, use_gt=False, mean_conv=0, plme=False,
        aug_data=None, raw_data=(cv_images_30, cv_landmarks_30), n_pert=30, 
        stop_after=-1, prefix='', hp_f=None):
    
    errors = []
    lmw_errors = []
    l_errors = None
    
    if aug_data is None:
        aug_data = raw_data
    
    for i in range(len(raw_data[0])):
        train = aug_data[0][i][0]
        train_lms = aug_data[1][i][0]
        test = raw_data[0][i][1]
        test_lms = raw_data[1][i][1]
        printb('Starting fold', i + 1)
        n = prefix + 'fold_' + str(i + 1) + '_' + str(int(TIF)) + '.ert'
        print('fitters/' + n)
        if os.path.exists('fitters/' + n):
            if verbose:
                print('Loaded fitter', n)
            fitter = DlibWrapper('fitters/' + n, TIF=TIF)
        else:
            if verbose:
                print('Training fitter', n)
            fitter = DlibERT(train, scales=(1), TIF=TIF, verbose=verbose, n_perturbations=n_pert)
            if verbose:
                print('Saving fitter', n)
            fitter.save('fitters/' + n)
            # reload because results get better (?)
            fitter = DlibWrapper('fitters/' + n, TIF=TIF)
        if verbose:
            print('Testing fold', i + 1)
        if MPI or hp_f is not None:
            if hp_f is not None:
                inits = hp_f(i, use_gt=use_gt)
                m_error, a_errors = test_eval(fitter, test, initialisations=inits)
            else:
                initialisations = init_f(train_lms, init_n)
                m_error, a_errors = test_eval(fitter, test, initialisations=initialisations, mean_conv=mean_conv) 
        else:
            if plme:
                m_error, a_errors, l_errors = test_eval(fitter, test, plme=True)
            else:
                m_error, a_errors = test_eval(fitter, test)
        errors.extend(a_errors)
        if plme:
            lmw_errors.extend(l_errors)
        print('Mean norm. error'.ljust(25), m_error)
        print('Success rate'.ljust(25), len(np.where(np.array(a_errors) < 0.1)[0]) / len(a_errors))
        printb('Finished fold', i + 1)
        print()
        if stop_after > 0 and i + 1 == stop_after:
            break
    if plme:
        return errors, lmw_errors
    return errors

### Run

In [0]:
errors_30 = ERT(n_pert=30, prefix='30_', raw_data=(cv_images_30, cv_landmarks_30), verbose=True)
plot_errors([('0 to 30 degrees', errors_30)])

Starting fold 1
fitters/30_fold_1_0.ert
Training fitter 30_fold_1_0.ert


KeyboardInterrupt: ignored

In [0]:
errors_60 = ERT(n_pert=50, prefix='60_', raw_data=(cv_images_60, cv_landmarks_60), verbose=True)
plot_errors([('30 to 60 degrees', errors_60)])

In [0]:
errors_90 = ERT(n_pert=50, prefix='90_', raw_data=(cv_images_90, cv_landmarks_90), verbose=True)
plot_errors([('60 to 90 degrees', errors_90)])

In [0]:
path_fitter = os.path.join('fitters', '90_fold_1_0.ert')

fitter = DlibWrapper(path_fitter, TIF=False)
for image in images_60:
    open_cv_frame = image.as_PILImage().convert('RGB')
    open_cv_frame = np.array(open_cv_frame)
    h, w, _= np.shape(open_cv_frame)
    left = 0
    top = 0
    right = w
    bottom = h
    print('top: %d left: %d right: %d bottom: %d' %(left, top, right, bottom))
    bb = menpo.shape.bounding_box((top, left), (bottom, right))
    print('type bb: ', type(bb))
    print('type image: ', type(image))
    shape = fitter.fit_from_bb(image = image, bounding_box = bb).final_shape
    
    # draw lms
    for (y, x) in shape.as_vector().reshape((-1, 2)):
        print('type open cv frame: ', type(open_cv_frame))
        cv.circle(open_cv_frame, (x, y), 3, (182, 127, 59), thickness=-1, lineType=cv.LINE_AA)
        cv.circle(open_cv_frame, (x, y), 3, (0, 0, 0), lineType=cv.LINE_AA)
    
    open_cv_frame = cv.cvtColor(open_cv_frame,cv.COLOR_BGR2RGB)
    cv.imshow('vid', open_cv_frame)
    cv.waitKey(0)
    cv.destroyAllWindows()